# Comandos PIP 

In [1]:
%pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade Pillow

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pytesseract

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -qU langchain_community langchain_huggingface pymupdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
open-webui 0.5.12 requires langchain==0.3.7, but you have langchain 0.3.22 which is incompatible.
open-webui 0.5.12 requires langchain-community==0.3.7, but you have langchain-community 0.3.20 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


# Imports

In [5]:
import os, shutil
os.environ["TESSDATA_PREFIX"] = r'/usr/share/tesseract-ocr/4.00/tessdata'

from tqdm.notebook import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import matplotlib.pyplot as plt
from langchain_core.document_loaders import BaseLoader
from langchain_community.document_loaders.pdf import BasePDFLoader
from langchain_core.documents import Document as LCDocument
from pathlib import Path
import logging
from typing import Any, Dict, Iterator, Mapping, Optional, List, TYPE_CHECKING
from langchain_core.document_loaders import BaseBlobParser
from langchain_community.document_loaders.blob_loaders import Blob
import warnings
import fitz
from __future__ import annotations
import time
from datetime import timedelta
from pymupdf import FileDataError
from langchain.docstore.document import Document as LangChainDocument
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores.utils import DistanceStrategy
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate, ChatMessagePromptTemplate, ChatPromptTemplate
import pytesseract
from PIL import Image

# LLM 🟥

## Document Loader (PyMuPDF)

In [6]:
if TYPE_CHECKING:
    import fitz.fitz

class PyMuPDFCustomParser(BaseBlobParser):
    """Parse `PDF` using `PyMuPDF`."""

    def __init__(
        self,
        text_kwargs: Optional[Mapping[str, Any]] = None,
    ) -> None:
        """Initialize the parser.

        Args:
            text_kwargs: Keyword arguments to pass to ``fitz.Page.get_text()``.
        """
        self.text_kwargs = text_kwargs or {}

    def lazy_parse(self, blob: Blob) -> Iterator[LCDocument]:  # type: ignore[valid-type]
        """Lazily parse the blob."""

        import fitz

        with blob.as_bytes_io() as file_path:  # type: ignore[attr-defined]
            
                if blob.data is None:  # type: ignore[attr-defined]
                    doc = fitz.open(file_path)
                else:
                    doc = fitz.open(stream=file_path, filetype="pdf")

                yield from [
                    LCDocument(
                        page_content=self._get_page_content(doc, page, blob),
                        metadata=self._extract_metadata(doc, page, blob),
                    )
                    for page in doc
                ]
    def _get_page_content(
        self, doc: fitz.fitz.Document, page: fitz.fitz.Page, blob: Blob
    ) -> str:
        """
        Get the text of the page using PyMuPDF and RapidOCR and issue a warning
        if it is empty.
        """
        content = page.get_text(**self.text_kwargs)
        images = page.get_images(full=True) 
        
        # if images:
        #    #print("Realizando OCR na pag {pag}".format(pag = page))
        #    retorno = self._extract_images_from_page(doc, page, images) 
        #    if ( retorno is not None):
        #        content += retorno
        #    else:
        #        retorno = self.extract_txt_saved_img('/home/pedro-nca/images/pdf_images')
        #        content += retorno
                
            



            
        # if not content:
        #     warnings.warn(
        #         f"Warning: Empty content on page "
        #         f"{page.number} of document {blob.source}"
        #     )

        return content

    def _extract_metadata(
        self, doc: fitz.fitz.Document, page: fitz.fitz.Page, blob: Blob
    ) -> dict:
        """Extract metadata from the document and page."""
        return dict(
            {
                "source": blob.source,  # type: ignore[attr-defined]
                "file_path": blob.source,  # type: ignore[attr-defined]
                "page": page.number,
                "total_pages": len(doc),
            },
            **{
                k: doc.metadata[k]
                for k in doc.metadata
                if isinstance(doc.metadata[k], (str, int))
            },
        )

    # def _extract_images_from_page(
    #     self, doc: fitz.fitz.Document, page: fitz.fitz.Page, images
    # ) -> str:
    #     """Extract images from page and get the text with RapidOCR."""
    #     import fitz
    #     for item in images:
            
    #         xref = item[0]
    #         smask = item[1]
    #         if smask > 0:
    #             pix0 = fitz.Pixmap(doc.extract_image(xref)["image"])
    #             if pix0.alpha: 
    #                 pix0 = fitz.Pixmap(pix0, 0)

    #                 #print(str(page)[6])
    #                 #pix0.save(filename='/home/pedro-nca/images/pdf_images/image'+str(page)[6]+'.jpeg')
    #                 #pix._writeIMG(output, format_=1, jpg_quality=)
    #                 #print("=== Convertendo PDF para imagem: {img} ===".format(img=output))
    #                 #print("PYTE ==== {text}".format(text=text))
    #                 pdfdata = pix0.pdfocr_tobytes()  
    #                 ocrpdf = fitz.open("pdf", pdfdata)  
    #                 ocrtext = ocrpdf[0].get_text()  
                    
    #                 return ocrtext
    #             mask = fitz.Pixmap(doc.extract_image(smask)["image"])
            
    #         if "/ColorSpace" in doc.xref_object(xref, compressed=True):
    #             pix0 = fitz.Pixmap(doc, xref)
    #             pix0 = fitz.Pixmap(fitz.csRGB, pix0)
    #             save_path = '/home/pedro-nca/images/pdf_images/image'+str(page)[6]+'.png'
    #             pix0.save(filename=save_path)
                
                


            # if pix.colorspace is None:
            #     print("ENTRO PARA CONVERSÃO ")
            #     pix = fitz.Pixmap(fitz.csRGB, pix) 
            #print(pix.colorspace)
            # if pix.colorspace.name not in (fitz.csGRAY.name, fitz.csRGB.name, fitz.csCMYK.name):
            #     #print("Essa imagem pode ser uma máscara.")
            #     #pix = fitz.Pixmap(fitz.csRGB, pix)
            #     return ''
            # else:
    
    def extract_txt_saved_img(self, img_folder_path):
        for arq in os.listdir(img_folder_path):
            if arq.endswith('.png'):
                img_file_path = os.path.join(img_folder_path, arq)                    
                text_each_img =  str(((pytesseract.image_to_string(Image.open(img_file_path), lang="por"))))
                return text_each_img

                

In [7]:
class PyMuPDFCustomLoader(BasePDFLoader):
    def __init__(self,
        file_path: str,
        *,
        headers: Optional[Dict] = None,
        **kwargs: Any,):
        
        try:
            import fitz  # noqa:F401
        except ImportError:
            raise ImportError(
                "`PyMuPDF` package not found, please install it with "
                "`pip install pymupdf`"
            )
        super().__init__(file_path, headers=headers)
        self.text_kwargs = kwargs
    
    def _lazy_load(self, **kwargs: Any) -> Iterator[LCDocument]:
        # if kwargs:
        #     logger.warning(
        #         f"Received runtime arguments {kwargs}. Passing runtime args to `load`"
        #         f" is deprecated. Please pass arguments during initialization instead."
        #     )

        text_kwargs = {**self.text_kwargs, **kwargs}
        parser = PyMuPDFCustomParser(
            text_kwargs=text_kwargs
        )
        if self.web_path:
            blob = Blob.from_data(open(self.file_path, "rb").read(), path=self.web_path)  # type: ignore[attr-defined]
        else:
            blob = Blob.from_path(self.file_path)  # type: ignore[attr-defined]
        yield from parser.lazy_parse(blob)

    def load(self, **kwargs: Any) -> List[LCDocument]:
        return list(self._lazy_load(**kwargs))

    def lazy_load(self) -> Iterator[LCDocument]:
        yield from self._lazy_load()


## Funções de Utilidades

In [8]:
# TXT = []
# for arq in os.listdir('/home/pedro-nca/images/pdf_images'):
#     if arq.endswith('.png'):
#         img_file_path = os.path.join('/home/pedro-nca/images/pdf_images', arq)                    
#         text_each_img =  str(((pytesseract.image_to_string(Image.open(img_file_path), lang="por"))))
#         TXT.append(text_each_img)

In [9]:
def delete_files_in_folder(folder_path):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))

## 1 - Carregamento dos PDF's

In [10]:
#path_folder = "/home/pedro-nca/pdf_mock/"
path_folder = '/home/matheus/Benchmark/'

from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
DOC_DATABASE = []

for folder_name in os.listdir(path_folder):
    pdf_folder_path= path_folder+folder_name
    for arq in os.listdir(pdf_folder_path):
        if arq.endswith('.pdf'):
               try:
                    pdf_file_path = os.path.join(pdf_folder_path, arq)                    
                    loader = PyMuPDFCustomLoader(file_path=pdf_file_path)
                    #loader = PyMuPDFLoader(pdf_file_path, extract_images=True)
                    #loader = PyPDFLoader(file_path=pdf_file_path, extract_images=True)
                    #getTables(pdf_file_path)
                    DOC_DATABASE.extend(loader.load())                
                    #delete_files_in_folder('/home/pedro-nca/images/pdf_images')
               except FileDataError:
                     pass
           

            

#DOC_DATABASE
    

In [11]:
DOC_DATABASE

[Document(metadata={'source': '/home/matheus/Benchmark/eh/0761.001_Exame_POSCOMP_2023.pdf', 'file_path': '/home/matheus/Benchmark/eh/0761.001_Exame_POSCOMP_2023.pdf', 'page': 0, 'total_pages': 20, 'format': 'PDF 1.5', 'title': 'Of', 'author': 'Cristina Girardi', 'subject': '', 'keywords': '', 'creator': 'Microsoft® Word 2013', 'producer': 'Microsoft® Word 2013', 'creationDate': "D:20230915130750-03'00'", 'modDate': "D:20230918085208-03'00'", 'trapped': ''}, page_content=' \n \n \nEXAME NACIONAL PARA INGRESSO NA  \nPÓS-GRADUAÇÃO EM COMPUTAÇÃO/2023 \n \nEXAME POSCOMP 2023 \nLeia atentamente e cumpra rigorosamente as instruções que seguem, pois elas são parte integrante \ndas provas e das normas que regem esse certame. \n1. Certifique-se de que o ambiente de prova: \n- contém 70 (setenta) questões; \n2. Cada questão oferece 5 (cinco) alternativas de respostas, representadas pelas letras A, B, C, D e \nE, sendo apenas 1 (uma) a resposta correta. \n3. Será respeitado o tempo para realização

In [12]:
texto_questoes = []
for doc in DOC_DATABASE:
    texto_questoes.append(doc.page_content)  

In [13]:
texto_questoes = texto_questoes[1:]
texto_questoes

['761_POSCOMP_NS_DM \nExecução: Fundatec \nEXAME POSCOMP 2023 \n15/9/202313:07:48 \nMATEMÁTICA \n \nQUESTÃO 01 – Utilize o método de Eliminação de Gauss para resolver o sistema a seguir: \n \n-3x + y + z = 1 \nx – 2y + z = 4 \n-x + y − 3z = -7 \n \nA) (0, -1, 2) \nB) (0, 1, 0) \nC) (2, 0, 2) \nD) (2, 1, 2)  \nE) (-1, -1, -1) \n \n \nQUESTÃO 02 – Determine o valor de x para que o vetor (1, x, 5) ∈ R3 pertença ao subespaço          \n<(1, 2, 3), (1, 1, 1)> \n \nA) x = 0 \nB) x = -1 \nC) x = 1 \nD) x = 3 \nE) x = 7 \n \n \nQUESTÃO 03 – Determine o polinômio característico de J: \n  \nJ = (\n2\n−2\n1\n1\n−1\n1\n1\n−2\n2\n) \n \nA) 0 \nB) x \nC) (1 − x) \nD) (1 − x)² \nE) (1 − x)³ \n \n \nQUESTÃO 04 – Nos jogos da Mega-Sena, são sorteados a cada concurso 6 números no intervalo de \n1 a 60. Leva o prêmio quem acertar os 6 números sorteados. O apostador, ao fazer um jogo, pode \noptar por preencher um bilhete com 6, 7, 8 ou 9 números escolhidos. Se o apostador registra um \nbilhete com 8 núme

In [14]:
import re

In [15]:
import re

def separar_pergunta_opcoes(bloco_questao):
    """
    Processa um bloco individual de questão separando pergunta e opções
    
    Args:
        bloco_questao (str): Texto contendo uma questão completa
        
    Returns:
        dict: Dicionário com 'numero', 'pergunta' e 'opcoes'
    """
    # Padrão para capturar o número da questão e a pergunta (incluindo múltiplas linhas)
    padrao_numero = re.compile(
        r'^QUESTÃO (\d+) – (.*?)(?=\n\s*[A-E]\))',
        re.DOTALL
    )
    match = padrao_numero.search(bloco_questao)
    
    if not match:
        return None
    
    numero = match.group(1)
    pergunta = match.group(2).strip()
    
    # Captura todas as opções (incluindo múltiplas linhas)
    padrao_opcoes = re.compile(
        r'([A-E]\)\s+.*?(?=\n[A-E]\)|\Z))',
        re.DOTALL
    )
    opcoes = padrao_opcoes.findall(bloco_questao)
    
    # Limpeza das opções: remove quebras de linha e espaços extras
    opcoes = [re.sub(r'\s+', ' ', opcao.strip()) for opcao in opcoes]
    
    return {
        'numero': numero,
        'pergunta': pergunta,
        'opcoes': opcoes
    }

def processar_textos_completos(lista_textos):
    """
    Processa uma lista de textos com múltiplas questões
    
    Args:
        lista_textos (list): Lista contendo textos com várias questões
        
    Returns:
        list: Lista de dicionários com questões processadas
    """
    todas_questoes = []
    
    for texto in lista_textos:
        # Divide o texto em blocos de questões usando "QUESTÃO" como marcador
        blocos_questoes = re.split(r'\n(?=QUESTÃO \d+)', texto.strip())
        
        for bloco in blocos_questoes:
            bloco = bloco.strip()
            if not bloco:
                continue
            questao = separar_pergunta_opcoes(bloco)
            if questao:
                todas_questoes.append(questao)
    
    return todas_questoes

In [16]:
# def separar_pergunta_opcoes(bloco_questao):
#     """
#     Processa um bloco individual de questão separando pergunta e opções.
    
#     Args:
#         bloco_questao (str): Texto contendo uma questão completa.
        
#     Returns:
#         dict: Dicionário com 'numero', 'pergunta' e 'opcoes'.
#     """
#     # Captura número da questão e a pergunta
#     padrao_numero_pergunta = r"^(\d+)\n(.+?)(?=\n\(A\))"
#     match = re.search(padrao_numero_pergunta, bloco_questao, re.DOTALL)
    
#     if not match:
#         return None
    
#     numero = match.group(1)
#     pergunta = match.group(2).strip()
    
#     # Captura todas as opções (A, B, C, D) e mantém o formato correto
#     padrao_opcoes = r"\((A|B|C|D)\)\s+(.*?)(?=\n\(A\)|\n\(B\)|\n\(C\)|\n\(D\)|$)"
#     opcoes = re.findall(padrao_opcoes, bloco_questao, re.DOTALL)
    
#     # Formata as opções corretamente
#     opcoes_formatadas = [f"{letra}) {texto.strip()}" for letra, texto in opcoes]
    
#     return {
#         'numero': numero,
#         'pergunta': pergunta,
#         'opcoes': opcoes_formatadas
#     }

# def processar_textos_completos(lista_textos):
#     """
#     Processa uma lista de textos com múltiplas questões.
    
#     Args:
#         lista_textos (list): Lista contendo textos com várias questões.
        
#     Returns:
#         list: Lista de dicionários com questões processadas.
#     """
#     todas_questoes = []
    
#     for texto in lista_textos:
#         # Divide cada questão com base em números no início da linha
#         blocos_questoes = re.split(r"\n(?=\d+\n)", texto.strip())
        
#         for bloco in blocos_questoes:
#             questao = separar_pergunta_opcoes(bloco)
#             if questao:
#                 todas_questoes.append(questao)
    
#     return todas_questoes


In [17]:
# Processamento
questoes = processar_textos_completos(texto_questoes)

# Exibindo os resultados
for questao in questoes:
    print(f"\nQUESTÃO {questao['numero']}:")
    print(questao['pergunta'])
    print("\nOPÇÕES:")
    for opcao in questao['opcoes']:
        print(opcao)
    print("-" * 50)


QUESTÃO 01:
Utilize o método de Eliminação de Gauss para resolver o sistema a seguir: 
 
-3x + y + z = 1 
x – 2y + z = 4 
-x + y − 3z = -7

OPÇÕES:
A) (0, -1, 2)
B) (0, 1, 0)
C) (2, 0, 2)
D) (2, 1, 2)
E) (-1, -1, -1)
--------------------------------------------------

QUESTÃO 02:
Determine o valor de x para que o vetor (1, x, 5) ∈ R3 pertença ao subespaço          
<(1, 2, 3), (1, 1, 1)>

OPÇÕES:
A) x = 0
B) x = -1
C) x = 1
D) x = 3
E) x = 7
--------------------------------------------------

QUESTÃO 03:
Determine o polinômio característico de J: 
  
J = (
2
−2
1
1
−1
1
1
−2
2
)

OPÇÕES:
A) 0
B) x
C) (1 − x)
D) (1 − x)²
E) (1 − x)³
--------------------------------------------------

QUESTÃO 04:
Nos jogos da Mega-Sena, são sorteados a cada concurso 6 números no intervalo de 
1 a 60. Leva o prêmio quem acertar os 6 números sorteados. O apostador, ao fazer um jogo, pode 
optar por preencher um bilhete com 6, 7, 8 ou 9 números escolhidos. Se o apostador registra um 
bilhete com 8 números 

In [18]:
import pandas as pd
import numpy as np
import csv

In [19]:
df = pd.DataFrame(questoes)
df

,numero,pergunta,opcoes
0,01,Utilize o método de Eliminação de Gauss para r...,"[A) (0, -1, 2), B) (0, 1, 0), C) (2, 0, 2), D)..."
1,02,"Determine o valor de x para que o vetor (1, x,...","[A) x = 0, B) x = -1, C) x = 1, D) x = 3, E) x..."
2,03,Determine o polinômio característico de J: \n ...,"[A) 0, B) x, C) (1 − x), D) (1 − x)², E) (1 − ..."
3,04,"Nos jogos da Mega-Sena, são sorteados a cada c...","[A) 56, B) 8!, C) 6!, D) 28, E) 72]"
4,05,"Recentemente, com a pandemia de Covid-19, houv...","[A) 1024, B) 10, C) 100, D) 512, E) 20]"
...,...,...,...
65,66,Qual dispositivo atua somente nas camadas físi...,"[A) Hub., B) Roteador., C) Repetidor., D) Gate..."
66,67,"Considere um pacote de p bytes, enviados por u...","[A) c/b, B) b/c, C) 8p/b, D) d/c, E) d/c + b/c]"
67,68,Uma transação cliente se torna distribuída se ...,"[A) plana, B) aninhada, C) coordenadora, D) em..."
68,69,"Em sistemas operacionais distribuídos, é corre...","[A) No escalonamento não preemptivo, uma threa..."


In [105]:
df.loc[19, 'pergunta'] = 'Para responder à questao, utilize a seguinte lista de dados, correspondente ao número de segundos que um software precisa para resolver um cálculo matemático: 60; 66; 77; 70; 66; 68; 57; 70; 66; 52; 75; 65; 69; 71; 58; 66; 67; 74; 61; 63; 69; 80; 59; 66; 70; 67; 78; 75; 64; 71; 81; 62; 64; 69; 68; 72; 83; 56; 65; 74; 67; 54; 65; 65; 69; 61; 67; 73; 57; 62; 67; 68; 63; 67; 71; 68; 76; 61; 62; 63; 76; 61; 67; 67; 64; 72; 64; 73; 79; 58; 67; 71; 68; 59; 69; 70; 66; 62; 63; 66; O tempo médio aproximado, que o software leva para realizar os cálculos é:'

In [117]:
df.loc[53].pergunta

'Analise as matrizes de transformações geométricas A e B. \n \n𝑨:     [\n𝑥′\n𝑦′\n1\n] = [\ncos (𝜃)\n−𝑠𝑒𝑛(𝜃)\n0\n𝑠𝑒𝑛(𝜃)\ncos (𝜃)\n0\n0\n0\n1\n] ∙[\n𝑥\n𝑦\n1\n] \n \n𝑩:     [\n𝑥′\n𝑦′\n1\n] = [\n𝐸𝑥\n0\n𝑇𝑥\n0\n𝐸𝑦\n𝑇𝑦\n0\n0\n1\n] ∙[\n𝑥\n𝑦\n1\n] \n \nConsiderando que 𝐸𝑥 e 𝐸𝑦 são fatores de escala, 𝑇𝑥 e 𝑇𝑦 são fatores de translação e que 𝜃 representa \num ângulo de rotação, analise as assertivas a seguir: \n \nI. A transformação 𝑨 rotaciona o objeto com pivô no centro de massa do próprio objeto. \nII. A transformação 𝑩 primeiro escala e depois translada o objeto. \n \nQuais estão corretas?'

In [136]:
# Nova linha como DataFrame
nova_linha = pd.DataFrame({'numero': [46], 'pergunta': ['''Analise as assertivas abaixo sobre estruturas em linguagens de programação: I. Uma estrutura é um conjunto de uma ou mais variáveis agrupadas sob um único nome, de forma a facilitar a sua referência.
II. A declaração de uma estrutura corresponde unicamente à definição de um novo tipo (isto é, da sua estrutura), e não à declaração de variáveis do tipo da estrutura.
III. Uma estrutura pode conter, na sua definição, variáveis simples, vetores, ponteiros ou mesmo outras estruturas.
IV. As estruturas permitem agrupar diversos componentes em uma única variável, que podem ser definidos com tipos distintos.
Quais estão corretas?'''], 'opcoes': [["A) (W+X)*( W̅ + X̅ + Y + Z)*( Y̅ + Z̅)",
"B) (W+ X̅+ Z̅)*(W+X+ Z̅ )*(W̅ + X̅ )*(Y̅ + Z)",
"C) (W̅ + X)*(W+Y+ Z̅)*(W+Y̅ +Z)* *(W+X̅ + Y̅ )",
"D) (W+X)*(W̅ + Y̅ + Z̅)",
"E) (W+X)*(W̅ + Y̅ + Z̅)*(W̅ + X̅ + Y + Z)"]]})

# Dividir e concatenar
df_final = pd.concat([
    df.iloc[:45], 
    nova_linha, 
    df.iloc[45:]
], ignore_index=True)

In [138]:
df_final.tail(40)
df = df_final

In [141]:
df.tail(40)
df = df[['numero','pergunta','opcoes']]

In [20]:
df.to_csv(
    'poscomp-23.csv',
    index=False,          # Não incluir índice
    sep=',',              # Delimitador padrão (vírgula)
    quoting=csv.QUOTE_ALL, # Forçar aspas em todos os campos
    quotechar='"',        # Usar aspas duplas para encapsular
    escapechar='\\',      # Escapar caracteres especiais se necessário
    encoding='utf-8'      # Codificação para preservar caracteres especiais
)
df

,numero,pergunta,opcoes
0,01,Utilize o método de Eliminação de Gauss para r...,"[A) (0, -1, 2), B) (0, 1, 0), C) (2, 0, 2), D)..."
1,02,"Determine o valor de x para que o vetor (1, x,...","[A) x = 0, B) x = -1, C) x = 1, D) x = 3, E) x..."
2,03,Determine o polinômio característico de J: \n ...,"[A) 0, B) x, C) (1 − x), D) (1 − x)², E) (1 − ..."
3,04,"Nos jogos da Mega-Sena, são sorteados a cada c...","[A) 56, B) 8!, C) 6!, D) 28, E) 72]"
4,05,"Recentemente, com a pandemia de Covid-19, houv...","[A) 1024, B) 10, C) 100, D) 512, E) 20]"
...,...,...,...
65,66,Qual dispositivo atua somente nas camadas físi...,"[A) Hub., B) Roteador., C) Repetidor., D) Gate..."
66,67,"Considere um pacote de p bytes, enviados por u...","[A) c/b, B) b/c, C) 8p/b, D) d/c, E) d/c + b/c]"
67,68,Uma transação cliente se torna distribuída se ...,"[A) plana, B) aninhada, C) coordenadora, D) em..."
68,69,"Em sistemas operacionais distribuídos, é corre...","[A) No escalonamento não preemptivo, uma threa..."
